In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from mapie.risk_control import MultiClassificationController
from mapie.risk_control.methods import ltt_procedure
from mapie.risk_control.risks import build_precision_ovr
from mapie.utils import check_valid_ltt_params_index
from sklearn.metrics import average_precision_score
import plotly.express as px
import plotly.graph_objects as go


#X_toy, y_toy = make_gaussian_quantiles(
#            n_samples=1000,
#            n_features=2,
#           n_classes=5,
#            #weights=[0.5, 0.5],
#        )



# faire lin 0,9 et on descend
# fix_sequence_testing .85 => 0.5

In [3]:
def run_exp_multiclasse(n_classes =  3,num_params = 10,target_level = 0.9, show_plot = True, data_per_classes=1000, fwer_method="bonferroni_holm"): 
    X_toy, y_toy = make_classification(
                n_samples=data_per_classes * n_classes,
                n_features=10,
                n_informative=10,
                n_redundant=0,
                n_repeated=0,
                n_classes=n_classes,
                n_clusters_per_class=1,
                #weights=[0.5, 0.5],
                flip_y=0,
                random_state=42
            )
    lr = LogisticRegression()
    clf = LogisticRegression().fit(X_toy, y_toy)

    # x and y given as array_like objects

    if show_plot:
        fig = px.scatter(x=X_toy[:,0], y=X_toy[:,1],color=y_toy)
        fig.show()
    # For split_fixed_sequence, the fixed-sequence ordering is learned starting
    # from the first entry of list_predict_params, so we scan from high to low
    # thresholds instead of low to high.
    list_predict_params = (
        np.linspace(0.95, 0.05, num_params)
        if fwer_method == "split_fixed_sequence"
        else np.linspace(0.05, 0.95, num_params)
    )
    risks = build_precision_ovr(np.unique(y_toy))
    mapie_clf = MultiClassificationController(predict_function=clf.predict_proba, target_level=target_level,risks=risks,
                                            confidence_level = 0.9,list_predict_params=list_predict_params,
                                            fwer_method=fwer_method,
                                            risk_combination_method=1)#
    # When fwer_method="split_fixed_sequence", the parameter order must be learned
    # on data independent from the calibration set (see learn_fixed_sequence_order docstring).
    if mapie_clf.fwer_method == "split_fixed_sequence":
        X_learn, X_calibrate, y_learn, y_calibrate = train_test_split(
            X_toy, y_toy, test_size=0.5, random_state=42, stratify=y_toy
        )
        mapie_clf.learn_fixed_sequence_order(X_learn, y_learn)
    else:
        X_calibrate = X_toy
        y_calibrate = y_toy
    y_calibrate_ = np.asarray(y_calibrate, dtype=int)
    mapie_clf.risk_combination_method = mapie_clf._check_risk_combination_method(
        mapie_clf.risk_combination_method, y_calibrate
    )
    original_params = mapie_clf._predict_params
    if mapie_clf.fwer_method == "split_fixed_sequence":
        if mapie_clf._learned_fixed_sequence is None:
            raise ValueError(
                "You must call 'learn_fixed_sequence_order' before 'calibrate' "
                "when using fwer_method='split_fixed_sequence'."
            )
        mapie_clf._predict_params = mapie_clf._learned_fixed_sequence

    predictions_per_param = mapie_clf._get_predictions_per_param(
        X_calibrate, mapie_clf._predict_params, is_calibration_step=True
    )
    # populate multirisks
    class_labels = np.unique(y_calibrate)
    # broadcast one _alpha_level per class.
    mapie_clf._alpha = np.repeat(mapie_clf._alpha, len(class_labels))
    risk_values, eff_sample_sizes = mapie_clf._get_risk_values_and_eff_sample_sizes(
        y_calibrate_, predictions_per_param, mapie_clf._risk
    )
    (valid_index, p_values) = ltt_procedure(
                risk_values,
                np.expand_dims(mapie_clf._alpha, axis=1),
                mapie_clf._delta,
                eff_sample_sizes,
                binary = True,
                fwer_method=mapie_clf.fwer_method,
            )
    return len(valid_index[0])

In [4]:
n_classes_range = np.arange(3, 11)  # 3 à 10
num_params_fixed = 90  # au-dela de ~80-90, augmenter num_params ne change plus le resultat
data_per_classes_range = np.array([100, 200, 300, 400, 500, 600, 700, 800, 900, 1000])  # 100 à 1000 (pas de 100) + 2000
target_level_range = np.round(np.arange(0.65, 0.95, 0.05), 2)  # 0.65 à 0.9 (pas de 0.05)
fwer_method_bonferroni_holm = "bonferroni_holm"

results = np.full(
    (len(n_classes_range), len(data_per_classes_range), len(target_level_range)),
    np.nan,
)

for i, n_classes in enumerate(n_classes_range):
    for j, data_per_classes in enumerate(data_per_classes_range):
        for k, target_level in enumerate(target_level_range):
            results[i, j, k] = run_exp_multiclasse(
                n_classes=n_classes,
                num_params=num_params_fixed,
                target_level=target_level,
                show_plot=False,
                data_per_classes=data_per_classes,
                fwer_method=fwer_method_bonferroni_holm,
            )


# Heatmap facettee : un facet par n_classes, x = target_level, y = data_per_classes,
# couleur = nb de parametres valides trouves par LTT. Les cellules a 0 sont
# masquees (NaN) pour ne pas les afficher.
grid_n_classes, grid_data_per_classes, grid_target_level = np.meshgrid(
    n_classes_range, data_per_classes_range, target_level_range, indexing="ij"
)

df_results = pd.DataFrame({
    "n_classes": grid_n_classes.ravel(),
    "data_per_classes": grid_data_per_classes.ravel(),
    "target_level": grid_target_level.ravel(),
    "nb_params_valides": np.where(results.ravel() == 0, np.nan, results.ravel()),
})

fig = px.density_heatmap(
    df_results,
    x="target_level",
    y="data_per_classes",
    z="nb_params_valides",
    histfunc="avg",
    facet_col="n_classes",
    facet_col_wrap=4,
    color_continuous_scale="Viridis",
    labels={"nb_params_valides": "nb params valides"},
    title=f"Nombre de parametres valides; (num_params_fixed => 90, fwer_method = {fwer_method_bonferroni_holm})",
)
fig.update_layout(height=700)
fig.show()

In [1]:
# Meme sweep, mais avec fwer_method="split_fixed_sequence" : contrairement a
# bonferroni-holm, cette methode necessite d'apprendre l'ordre des parametres
# (learn_fixed_sequence_order) sur des donnees independantes du jeu de
# calibration avant de calibrer. C'est gere dans run_exp_multiclasse.
fwer_method_fixed_sequence = "split_fixed_sequence"

results_fixed_sequence = np.full(
    (len(n_classes_range), len(data_per_classes_range), len(target_level_range)),
    np.nan,
)

for i, n_classes in enumerate(n_classes_range):
    for j, data_per_classes in enumerate(data_per_classes_range):
        for k, target_level in enumerate(target_level_range):
            results_fixed_sequence[i, j, k] = run_exp_multiclasse(
                n_classes=n_classes,
                num_params=num_params_fixed,
                target_level=target_level,
                show_plot=False,
                data_per_classes=data_per_classes,
                fwer_method=fwer_method_fixed_sequence,
            )


df_results_fixed_sequence = pd.DataFrame({
    "n_classes": grid_n_classes.ravel(),
    "data_per_classes": grid_data_per_classes.ravel(),
    "target_level": grid_target_level.ravel(),
    "nb_params_valides": np.where(
        results_fixed_sequence.ravel() == 0, np.nan, results_fixed_sequence.ravel()
    ),
})

fig = px.density_heatmap(
    df_results_fixed_sequence,
    x="target_level",
    y="data_per_classes",
    z="nb_params_valides",
    histfunc="avg",
    facet_col="n_classes",
    facet_col_wrap=4,
    color_continuous_scale="Viridis",
    labels={"nb_params_valides": "nb params valides"},
    title=f"Nombre de parametres valides (LTT) selon n_classes, data_per_classes, target_level (num_params_fixed => 90, fwer_method = {fwer_method_fixed_sequence})",
)
fig.update_layout(height=700)
fig.show()

NameError: name 'np' is not defined